# Executive Brief Generator

## Business question

How can we produce a concise, evidence-backed weekly executive brief from computed complaint intelligence metrics?

In [ ]:
from pathlib import Path

import pandas as pd

scores_path = Path("reports/sample/complaint_scores.csv")
terms_path = Path("reports/sample/topic_terms.csv")
if not scores_path.exists() or not terms_path.exists():
    raise FileNotFoundError(
        "Run `cx-complaints analyze --output-dir reports/sample` first."
    )

scores = pd.read_csv(scores_path)
topic_terms = pd.read_csv(terms_path)
scores["date_received"] = pd.to_datetime(scores["date_received"], errors="coerce")
scores = scores.dropna(subset=["date_received"]).copy()
scores["week"] = scores["date_received"].dt.to_period("W").astype(str)

## 1. Build weekly KPI context

In [ ]:
weekly = (
    scores.groupby("week")
    .agg(
        complaints=("complaint_id", "size"),
        high_risk=("risk_level", lambda s: (s == "high").sum()),
        medium_or_high=("risk_level", lambda s: s.isin(["medium", "high"]).sum()),
        avg_risk_score=("risk_score", "mean"),
        untimely=("timely", lambda s: (s.fillna("").str.lower() != "yes").sum()),
    )
    .reset_index()
    .sort_values("week")
)
weekly["high_risk_share_pct"] = (weekly["high_risk"] / weekly["complaints"] * 100).round(1)
weekly["untimely_share_pct"] = (weekly["untimely"] / weekly["complaints"] * 100).round(1)
weekly["avg_risk_score"] = weekly["avg_risk_score"].round(2)
weekly.tail(4)

In [ ]:
if len(weekly) < 2:
    raise ValueError("Need at least two weeks of data for a weekly brief.")

current = weekly.iloc[-1]
previous = weekly.iloc[-2]

delta = {
    "complaints": int(current["complaints"] - previous["complaints"]),
    "high_risk_share_pct": round(
        current["high_risk_share_pct"] - previous["high_risk_share_pct"],
        1,
    ),
    "untimely_share_pct": round(
        current["untimely_share_pct"] - previous["untimely_share_pct"],
        1,
    ),
    "avg_risk_score": round(current["avg_risk_score"] - previous["avg_risk_score"], 2),
}

current, previous, delta

## 2. Topic movement and priority queue context

In [ ]:
topic_labels = (
    topic_terms.sort_values(["topic_id", "rank"])
    .groupby("topic_id")["term"]
    .apply(lambda s: ", ".join(s.head(4).astype(str)))
    .rename("topic_label")
    .reset_index()
)

topic_weekly = scores.groupby(["week", "topic_id"]).size().rename("complaints").reset_index()
latest_week = current["week"]
prev_week = previous["week"]

latest_topic = topic_weekly[topic_weekly["week"] == latest_week][["topic_id", "complaints"]]
latest_topic = latest_topic.rename(columns={"complaints": "latest_complaints"})
prev_topic = topic_weekly[topic_weekly["week"] == prev_week][["topic_id", "complaints"]]
prev_topic = prev_topic.rename(columns={"complaints": "prev_complaints"})

topic_move = topic_labels.merge(latest_topic, on="topic_id", how="left")\
    .merge(prev_topic, on="topic_id", how="left")
topic_move[["latest_complaints", "prev_complaints"]] = (
    topic_move[["latest_complaints", "prev_complaints"]].fillna(0)
)
topic_move["delta_complaints"] = topic_move["latest_complaints"] - topic_move["prev_complaints"]
topic_move = topic_move.sort_values("delta_complaints", ascending=False)
topic_move.head(8)

In [ ]:
queue = scores.copy()
queue["timely_is_yes"] = queue["timely"].fillna("").str.lower() == "yes"
queue["weak_response"] = queue["company_response"].fillna("").str.lower().isin(
    ["closed", "closed with explanation", "in progress"]
)

priority_queue = queue[
    (queue["risk_level"] == "high")
    & (~queue["timely_is_yes"])
    & (queue["weak_response"])
].copy()
priority_queue_week = priority_queue[priority_queue["week"] == latest_week]

queue_stats = {
    "priority_queue_count": int(len(priority_queue_week)),
    "priority_queue_share_pct": round(
        len(priority_queue_week) / max(int(current["complaints"]), 1) * 100,
        1,
    ),
}
queue_stats

## 3. Generate draft executive brief

In [ ]:
top_rising_topics = topic_move.head(3)
topic_lines = []
for _, row in top_rising_topics.iterrows():
    topic_lines.append(
        f"- Topic {int(row['topic_id'])} ({row['topic_label']}): "
        f"{int(row['delta_complaints'])} complaint change WoW."
    )

brief = [
    f"# Weekly Executive Brief ({latest_week})",
    "",
    "## What changed",
    "",
    f"- Complaints: {int(current['complaints']):,} ({delta['complaints']:+,} WoW).",
    f"- High-risk share: {current['high_risk_share_pct']:.1f}% "
    f"({delta['high_risk_share_pct']:+.1f} pp WoW).",
    f"- Untimely share: {current['untimely_share_pct']:.1f}% "
    f"({delta['untimely_share_pct']:+.1f} pp WoW).",
    f"- Average risk score: {current['avg_risk_score']:.2f} ({delta['avg_risk_score']:+.2f} WoW).",
    "",
    "## Emerging topic movement",
    "",
    *topic_lines,
    "",
    "## Operational queue",
    "",
    "- Priority queue (high-risk + untimely + weak response): "
    f"{queue_stats['priority_queue_count']:,} complaints "
    f"({queue_stats['priority_queue_share_pct']:.1f}% of weekly volume).",
    "",
    "## Recommended actions",
    "",
    "1. Review the top-priority queue items within 24 hours.",
    "2. Assign owners for the top rising topics and validate root causes.",
    "3. Track whether high-risk and untimely shares normalize next week.",
]

brief_text = "\n".join(brief)
print(brief_text)

In [ ]:
# Save draft brief
output_dir = Path("reports/notebook")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "weekly_executive_brief.md"
output_path.write_text(brief_text, encoding="utf-8")
output_path